# 19A4C — Generic Year-wise Frozen Cycle-25 AIA Inference

Run frozen inference for one Cycle-25 year selected through `CYCLE25_YEAR`.

No retraining, recalibration, threshold reselection, or annual performance scoring is allowed.
The notebook only verifies staged data, loads the frozen 19A2 CNN-GRU, applies the frozen 19A3
Platt transform and threshold, and saves a prediction shard plus protocol record.


In [1]:
from pathlib import Path
import hashlib, json, os, time
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

HOME = Path.home()
YEAR = int(os.environ.get("CYCLE25_YEAR", "0"))

EXPECTED = {
    2022: {"targets": 9999, "positives": 119, "unique_objects": 10913},
    2023: {"targets": 12392, "positives": 278, "unique_objects": 13566},
    2024: {"targets": 12553, "positives": 1316, "unique_objects": 13831},
    2025: {"targets": 9281, "positives": 512, "unique_objects": 10879},
}
if YEAR not in EXPECTED:
    raise RuntimeError(f"CYCLE25_YEAR must be one of {sorted(EXPECTED)}, got {YEAR}")

TARGET_MAP = HOME / "aia19_cycle25_staging_prep" / "target_maps" / f"cycle25_{YEAR}_targets_local_paths.csv.gz"
BASE = HOME / "aia19_cnn_gru_20260917"
MODEL_PATH = BASE / "models" / "cnn_gru_cycle24_final_refit.pt"
NORM_PATH = BASE / "normalisation.json"
CAL_DIR = HOME / "aia19_calibration_threshold_20260918"
CAL_PATH = CAL_DIR / "platt_calibrator.json"
THR_PATH = CAL_DIR / "operating_threshold.json"
OUT = HOME / "aia19_cycle25_predictions"
OUT.mkdir(parents=True, exist_ok=True)

EXPECTED_MODEL_SHA256 = "11dc35e089101c8b79d2a6ba6f82d02cdb470d583552cad73e6071016c2a2d76"
EXPECTED_CHANNELS = ["aia94","aia131","aia171","aia193","aia211","aia335"]

IMAGE_SIZE = 256
BATCH_SIZE = 8
NUM_WORKERS = 2
DROPOUT = 0.30
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for p in [TARGET_MAP, MODEL_PATH, NORM_PATH, CAL_PATH, THR_PATH]:
    assert p.exists(), p

print("YEAR:", YEAR)
print("DEVICE:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)


YEAR: 2022
DEVICE: cuda
GPU: NVIDIA L4


In [2]:
def sha256_file(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

model_sha = sha256_file(MODEL_PATH)
assert model_sha == EXPECTED_MODEL_SHA256, model_sha

df = pd.read_csv(TARGET_MAP)
exp = EXPECTED[YEAR]

assert len(df) == exp["targets"], len(df)
assert int(df["label_48h_final"].sum()) == exp["positives"]
assert set(df["stored_year"].unique()) == {YEAR}
assert set(df["role"].unique()) == {"independent_cycle25_test"}

print("MODEL_SHA256:", model_sha)
print("Targets:", len(df))
print("Positives:", int(df["label_48h_final"].sum()))
print("Regions:", df["region_component_id"].nunique())


MODEL_SHA256: 11dc35e089101c8b79d2a6ba6f82d02cdb470d583552cad73e6071016c2a2d76
Targets: 9999
Positives: 119
Regions: 190


In [3]:
LOCAL_COLS = ["local_tminus288", "local_tminus192", "local_tminus96"]
all_paths = pd.unique(pd.concat([df[c] for c in LOCAL_COLS], ignore_index=True))
assert len(all_paths) == exp["unique_objects"], len(all_paths)

missing, zero = [], []
for p in all_paths:
    q = Path(p)
    if not q.exists():
        missing.append(str(q))
    elif q.stat().st_size <= 0:
        zero.append(str(q))

print("Expected unique staged objects:", exp["unique_objects"])
print("Existing:", exp["unique_objects"] - len(missing))
print("Missing:", len(missing))
print("Zero-byte:", len(zero))

if missing or zero:
    raise RuntimeError(f"{YEAR} staged payload integrity failed.")

canary_idx = np.linspace(0, len(all_paths)-1, num=min(5, len(all_paths)), dtype=int)
for i in canary_idx:
    p = all_paths[i]
    with np.load(p, allow_pickle=False) as z:
        x = z["x"]
        channels = [str(v) for v in z["channels"].tolist()]
        assert x.shape == (512,512,6)
        assert x.dtype == np.float32
        assert channels == EXPECTED_CHANNELS
        assert np.isfinite(x).all()

print(f"{YEAR} payload integrity/readability PASS")


Expected unique staged objects: 10913
Existing: 10913
Missing: 0
Zero-byte: 0


2022 payload integrity/readability PASS


In [4]:
norm = json.loads(NORM_PATH.read_text())
assert norm["channel_order"] == EXPECTED_CHANNELS
channel_scale = np.asarray(norm["channel_scale"], dtype=np.float32)
channel_mean = np.asarray(norm["channel_mean"], dtype=np.float32)
channel_std = np.asarray(norm["channel_std"], dtype=np.float32)

cal = json.loads(CAL_PATH.read_text())
thr = json.loads(THR_PATH.read_text())

PLATT_COEF = float(cal["coefficient"])
PLATT_INTERCEPT = float(cal["intercept"])
FROZEN_THRESHOLD = float(thr["threshold"])

assert abs(PLATT_COEF - 0.5846760189574352) < 1e-12
assert abs(PLATT_INTERCEPT - (-3.6728404851652265)) < 1e-12
assert abs(FROZEN_THRESHOLD - 0.030438695842933242) < 1e-15

print("Platt coefficient:", PLATT_COEF)
print("Platt intercept:", PLATT_INTERCEPT)
print("Frozen threshold:", FROZEN_THRESHOLD)


Platt coefficient: 0.5846760189574352
Platt intercept: -3.6728404851652265
Frozen threshold: 0.030438695842933242


In [5]:
class TemporalAIADataset(Dataset):
    def __init__(self, frame): self.df = frame.reset_index(drop=True)
    def __len__(self): return len(self.df)

    def _load(self, path):
        with np.load(path, allow_pickle=False) as z:
            x = z["x"]
            channels = [str(v) for v in z["channels"].tolist()]
            if x.shape != (512,512,6): raise ValueError((path, x.shape))
            if channels != EXPECTED_CHANNELS: raise ValueError((path, channels))
            if not np.isfinite(x).all(): raise ValueError(f"Nonfinite tensor: {path}")
            x = x.astype(np.float32, copy=False)

        x = np.arcsinh(x / channel_scale.reshape(1,1,6))
        x = (x - channel_mean.reshape(1,1,6)) / channel_std.reshape(1,1,6)
        t = torch.from_numpy(x).permute(2,0,1).contiguous()
        return F.interpolate(
            t.unsqueeze(0), size=(IMAGE_SIZE, IMAGE_SIZE),
            mode="bilinear", align_corners=False
        ).squeeze(0)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        x = torch.stack([
            self._load(r["local_tminus288"]),
            self._load(r["local_tminus192"]),
            self._load(r["local_tminus96"]),
        ])
        return x, int(r["label_48h_final"]), r["target_sample_id"], r["region_component_id"], int(r["HARPNUM"])

class ConvBlock(nn.Module):
    def __init__(self, cin, cout, drop=0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin,cout,3,stride=2,padding=1,bias=False),
            nn.BatchNorm2d(cout), nn.GELU(),
            nn.Conv2d(cout,cout,3,padding=1,bias=False),
            nn.BatchNorm2d(cout), nn.GELU(),
            nn.Dropout2d(drop) if drop else nn.Identity(),
        )
    def forward(self, x): return self.net(x)

class FrameCNN(nn.Module):
    def __init__(self, embed=256):
        super().__init__()
        self.enc = nn.Sequential(
            ConvBlock(6,32,.05), ConvBlock(32,64,.05),
            ConvBlock(64,128,.10), ConvBlock(128,192,.10),
            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(), nn.Linear(192,embed), nn.GELU(), nn.Dropout(DROPOUT)
        )
    def forward(self, x): return self.proj(self.enc(x))

class Model(nn.Module):
    def __init__(self, embed=256, hidden=192):
        super().__init__()
        self.frame = FrameCNN(embed)
        self.gru = nn.GRU(embed, hidden, batch_first=True)
        self.head = nn.Sequential(nn.LayerNorm(hidden), nn.Dropout(DROPOUT), nn.Linear(hidden,1))
    def forward(self, x):
        b,t,c,h,w = x.shape
        z = self.frame(x.reshape(b*t,c,h,w)).reshape(b,t,-1)
        _, hlast = self.gru(z)
        return self.head(hlast[-1]).squeeze(-1)

model = Model().to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()
print("Frozen model loaded.")


Frozen model loaded.


In [6]:
loader = DataLoader(
    TemporalAIADataset(df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
    drop_last=False,
)

records = []
t0 = time.time()

with torch.no_grad():
    for bi, (x, y, sid, region, harpnum) in enumerate(loader, 1):
        logits = model(x.to(DEVICE, non_blocking=True)).cpu().numpy().astype(float)

        raw_p = 1.0 / (1.0 + np.exp(-logits))
        calibrated_logit = PLATT_COEF * logits + PLATT_INTERCEPT
        calibrated_p = 1.0 / (1.0 + np.exp(-calibrated_logit))
        frozen_pred = (calibrated_p >= FROZEN_THRESHOLD).astype(int)

        for j in range(len(logits)):
            records.append({
                "target_sample_id": sid[j],
                "stored_year": YEAR,
                "region_component_id": region[j],
                "HARPNUM": int(harpnum[j]),
                "y_true": int(y[j]),
                "raw_logit": float(logits[j]),
                "raw_probability": float(raw_p[j]),
                "calibrated_probability": float(calibrated_p[j]),
                "frozen_threshold": FROZEN_THRESHOLD,
                "frozen_prediction": int(frozen_pred[j]),
            })

        if bi % 100 == 0 or bi == len(loader):
            print(f"{YEAR} inference batch {bi}/{len(loader)}", flush=True)

pred_df = pd.DataFrame(records)
assert len(pred_df) == exp["targets"]
assert pred_df["target_sample_id"].is_unique
assert int(pred_df["y_true"].sum()) == exp["positives"]
assert np.isfinite(pred_df[["raw_logit","raw_probability","calibrated_probability"]].to_numpy()).all()

elapsed = time.time() - t0
print("Inference rows:", len(pred_df))
print("Elapsed seconds:", elapsed)
print("No performance metrics computed.")


2022 inference batch 100/1250


2022 inference batch 200/1250


2022 inference batch 300/1250


2022 inference batch 400/1250


2022 inference batch 500/1250


...

2022 inference batch 1000/1250


2022 inference batch 1100/1250


2022 inference batch 1200/1250


2022 inference batch 1250/1250


Inference rows: 9999
Elapsed seconds: 958.8134613037109
No performance metrics computed.


In [7]:
PRED_PATH = OUT / f"cycle25_{YEAR}_frozen_predictions.csv.gz"
pred_df.to_csv(PRED_PATH, index=False, compression="gzip")
pred_sha = sha256_file(PRED_PATH)

protocol = {
    "status": f"CYCLE25_{YEAR}_FROZEN_AIA_PREDICTION_SHARD_COMPLETE_NO_METRICS",
    "year": YEAR,
    "targets": int(len(pred_df)),
    "positives": int(pred_df["y_true"].sum()),
    "base_model_sha256": model_sha,
    "normalisation_path": str(NORM_PATH),
    "platt_coefficient": PLATT_COEF,
    "platt_intercept": PLATT_INTERCEPT,
    "frozen_threshold": FROZEN_THRESHOLD,
    "prediction_file": str(PRED_PATH),
    "prediction_sha256": pred_sha,
    "model_weights_updated": False,
    "calibrator_refit": False,
    "threshold_reselected": False,
    "performance_metrics_computed": False,
    "cycle25_used_for_tuning": False,
    "scientific_clearance": False,
}

PROTO_PATH = OUT / f"cycle25_{YEAR}_protocol_record.json"
PROTO_PATH.write_text(json.dumps(protocol, indent=2) + "\n")

print(json.dumps(protocol, indent=2))
print("PREDICTION_SHA256:", pred_sha)
print(f"{YEAR}_FROZEN_INFERENCE_COMPLETE")


{
  "status": "CYCLE25_2022_FROZEN_AIA_PREDICTION_SHARD_COMPLETE_NO_METRICS",
  "year": 2022,
  "targets": 9999,
  "positives": 119,
  "base_model_sha256": "11dc35e089101c8b79d2a6ba6f82d02cdb470d583552cad73e6071016c2a2d76",
  "normalisation_path": "/home/abmoses2000/aia19_cnn_gru_20260917/normalisation.json",
  "platt_coefficient": 0.5846760189574352,
  "platt_intercept": -3.6728404851652265,
  "frozen_threshold": 0.030438695842933242,
  "prediction_file": "/home/abmoses2000/aia19_cycle25_predictions/cycle25_2022_frozen_predictions.csv.gz",
  "prediction_sha256": "5ce62b87fbea391020528983972fdd6b58514bca2f935cf1b08a94eb10e6d75b",
  "model_weights_updated": false,
  "calibrator_refit": false,
  "threshold_reselected": false,
  "performance_metrics_computed": false,
  "cycle25_used_for_tuning": false,
  "scientific_clearance": false
}
PREDICTION_SHA256: 5ce62b87fbea391020528983972fdd6b58514bca2f935cf1b08a94eb10e6d75b
2022_FROZEN_INFERENCE_COMPLETE
